In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [4]:
pip install mlflow dagshub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.1 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 58.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 72.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

In [6]:
import dagshub
import mlflow
import mlflow.sklearn

#dagshub.init(repo_owner='YOUR_DAGSHUB_USERNAME', repo_name='YOUR_REPO_NAME', mlflow=True)
dagshub.init(repo_owner='mkhak23', repo_name='ML_assignment2', mlflow=True)

Initialized MLflow to track repo "mkhak23/ML_assignment2"

Repository mkhak23/ML_assignment2 initialized!

In [7]:
identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')

In [8]:
from sklearn.model_selection import train_test_split

train = transaction.merge(identity, on = 'TransactionID', how = 'left')

X = train.drop(columns=["isFraud","TransactionID"])
y = train["isFraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {X_train.shape}")
print(f"Test size:  {X_test.shape}")

Train size: (472432, 432)
Test size:  (118108, 432)


# **Data Cleaning**

In [9]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
import pandas as pd

class DecisionTreeImputer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.imputer = SimpleImputer(strategy="median")

    def fit(self, X, y=None):
        self.columns_ = X.columns
        self.imputer.fit(X)
        return self

    def transform(self, X):
        X = X.copy()
        X_imputed = self.imputer.transform(X)
        return pd.DataFrame(X_imputed, columns=self.columns_, index=X.index)

# **Feature Engineering**

In [10]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd

class CategoricalOrdinalEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols=None):
        self.cols = cols
        self.encoder = OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1
        )

    def fit(self, X, y=None):
        X = X.copy()

        if self.cols is None:
            self.cols_ = X.select_dtypes(include=["object"]).columns.tolist()
        else:
            self.cols_ = self.cols

        X_cat = X[self.cols_].fillna("missing")

        self.encoder.fit(X_cat)

        return self

    def transform(self, X):
        X = X.copy()

        X_cat = X[self.cols_].fillna("missing")

        encoded = self.encoder.transform(X_cat)

        encoded_df = pd.DataFrame(
            encoded,
            columns=self.cols_,
            index=X.index
        )

        X = X.drop(columns=self.cols_)
        X = pd.concat([X, encoded_df], axis=1)

        return X

In [11]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

class FraudFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, uid_cols=("card1", "addr1")):
        self.uid_cols = uid_cols
        self.uid_means_ = None

    def fit(self, X, y=None):
        X = X.copy()

        uid = self._make_uid(X)

        self.uid_means_ = (
            pd.DataFrame({
                "uid": uid,
                "TransactionAmt": X["TransactionAmt"]
            })
            .groupby("uid")["TransactionAmt"]
            .mean()
        )

        return self

    def transform(self, X):
        X = X.copy()

        X["hour"] = (X["TransactionDT"] // 3600) % 24

        id_cols = [col for col in X.columns if col.startswith("id_")]
        if id_cols:
            X["identity_missing"] = X[id_cols].isna().sum(axis=1)
        else:
            X["identity_missing"] = 0

        uid = self._make_uid(X)

        uid_mean = uid.map(self.uid_means_)

        global_mean = self.uid_means_.mean()
        uid_mean = uid_mean.fillna(global_mean)

        X["uid_amt_mean"] = uid_mean

        X["uid_amt_diff"] = X["TransactionAmt"] - X["uid_amt_mean"]

        return X

    def _make_uid(self, X):
        uid = X[self.uid_cols[0]].astype(str)
        for col in self.uid_cols[1:]:
            uid += "_" + X[col].astype(str)
        return uid

# **Feature Selection**

In [12]:
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

class UselessFeatureDropper(BaseEstimator, TransformerMixin):
    def __init__(self,  variance_threshold=0.0):
        self.variance_threshold = variance_threshold
        self.cols_to_drop_ = []

    def fit(self, X, y=None):
        X = X.copy()
        self.cols_to_drop_ = []

        constant_cols = [
            col for col in X.columns
            if X[col].nunique(dropna=False) <= 1
        ]

        numeric_cols = X.select_dtypes(include=["number"]).columns

        near_zero_var_cols = [
            col for col in numeric_cols
            if X[col].var(skipna=True) <= self.variance_threshold
        ]

        self.cols_to_drop_ = list(set(
            constant_cols + near_zero_var_cols
        ))

        return self

    def transform(self, X):
        X = X.copy()
        return X.drop(columns=self.cols_to_drop_, errors="ignore")

In [18]:
import mlflow
import pandas as pd

mlflow.set_experiment("decision_tree_training")

with mlflow.start_run(run_name="cleaning_feature_engineering"):

    preprocessing_pipeline = Pipeline([
        ("drop_useless", UselessFeatureDropper()),
        ("features", FraudFeatureEngineer()),
        ("cat_enc", CategoricalOrdinalEncoder()),
        ("imputer", DecisionTreeImputer())
    ])

    X_processed = preprocessing_pipeline.fit_transform(X_train, y_train)

    mlflow.log_param("variance_threshold", 0.0)
    mlflow.log_param("drop_useless", True)
    mlflow.log_param("feature_hour", True)
    mlflow.log_param("feature_identity_missing", True)
    mlflow.log_param("feature_uid_amt_diff", True)
    mlflow.log_param("categorical_encoder", "OrdinalEncoder")
    mlflow.log_param("imputer", "median")

    mlflow.log_param("rows_before", X_train.shape[0])
    mlflow.log_param("cols_before", X_train.shape[1])

    mlflow.log_param("rows_after", X_processed.shape[0])
    mlflow.log_param("cols_after", X_processed.shape[1])

    dropped_cols = preprocessing_pipeline.named_steps["drop_useless"].cols_to_drop_
    mlflow.log_param("num_dropped_columns", len(dropped_cols))

    cat_cols = preprocessing_pipeline.named_steps["cat_enc"].cols_
    mlflow.log_param("num_categorical_cols", len(cat_cols))

    mlflow.log_param("missing_before", int(X_train.isna().sum().sum()))
    mlflow.log_param("missing_after", int(pd.DataFrame(X_processed).isna().sum().sum()))

    

🏃 View run cleaning_feature_engineering at: https://dagshub.com/mkhak23/ML_assignment2.mlflow/#/experiments/1/runs/d6bbb854e641459aaacff0053984e735
🧪 View experiment at: https://dagshub.com/mkhak23/ML_assignment2.mlflow/#/experiments/1


# **Decision Tree Model**

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, recall_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
import pandas as pd
import mlflow

mlflow.set_experiment("decision_tree_training")

param_list = [
    {"max_depth": 4, "min_samples_split": 50, "min_samples_leaf": 20},
    {"max_depth": 6, "min_samples_split": 100, "min_samples_leaf": 30},
    {"max_depth": 8, "min_samples_split": 200, "min_samples_leaf": 50},
]

results = []

for params in param_list:
    with mlflow.start_run(run_name="DecisionTree_training"):

        model = DecisionTreeClassifier(
            **params,
            class_weight="balanced",
            random_state=42
        )

        temp_pipeline = Pipeline([
            ("drop_useless", UselessFeatureDropper()),
            ("features", FraudFeatureEngineer()),
            ("cat_enc", CategoricalOrdinalEncoder()),
            ("imputer", DecisionTreeImputer()),
            ("model", model)
        ])

        temp_pipeline.fit(X_train, y_train)

        train_preds = temp_pipeline.predict_proba(X_train)[:, 1]
        train_y_pred = (train_preds >= 0.5).astype(int)

        val_preds = temp_pipeline.predict_proba(X_test)[:, 1]
        val_y_pred = (val_preds >= 0.5).astype(int)

        train_auc = roc_auc_score(y_train, train_preds)
        train_pr_auc = average_precision_score(y_train, train_preds)
        train_recall = recall_score(y_train, train_y_pred)
        train_f1 = f1_score(y_train, train_y_pred)

        val_auc = roc_auc_score(y_test, val_preds)
        val_pr_auc = average_precision_score(y_test, val_preds)
        val_recall = recall_score(y_test, val_y_pred)
        val_f1 = f1_score(y_test, val_y_pred)

        auc_gap = train_auc - val_auc

        if train_auc < 0.70 and val_auc < 0.70:
            fit_status = "underfitting"
        elif auc_gap > 0.05:
            fit_status = "overfitting"
        else:
            fit_status = "reasonable_fit"

        mlflow.log_params(params)

        mlflow.log_metric("train_roc_auc", train_auc)
        mlflow.log_metric("val_roc_auc", val_auc)
        mlflow.log_metric("auc_gap", auc_gap)

        mlflow.log_metric("train_pr_auc", train_pr_auc)
        mlflow.log_metric("val_pr_auc", val_pr_auc)

        mlflow.log_metric("train_recall", train_recall)
        mlflow.log_metric("val_recall", val_recall)

        mlflow.log_metric("train_f1", train_f1)
        mlflow.log_metric("val_f1", val_f1)

        mlflow.set_tag("fit_status", fit_status)

        results.append({
            **params,
            "val_roc_auc": val_auc,
        })

        print(params)
        print("Train AUC:", train_auc)
        print("Val AUC:", val_auc)
        print("Gap:", auc_gap)
        print("Status:", fit_status)
        print("-" * 40)

results_df = pd.DataFrame(results).sort_values("val_roc_auc", ascending=False)
results_df

{'max_depth': 4, 'min_samples_split': 50, 'min_samples_leaf': 20}
Train AUC: 0.8056542063970759
Val AUC: 0.8023836892506013
Gap: 0.0032705171464745275
Status: reasonable_fit
----------------------------------------
🏃 View run DecisionTree_training at: https://dagshub.com/mkhak23/ML_assignment2.mlflow/#/experiments/1/runs/d9f2d97c453b405fab87d91d3ab59d6c
🧪 View experiment at: https://dagshub.com/mkhak23/ML_assignment2.mlflow/#/experiments/1
{'max_depth': 6, 'min_samples_split': 100, 'min_samples_leaf': 30}
Train AUC: 0.8439624198669138
Val AUC: 0.8399687278872424
Gap: 0.0039936919796713655
Status: reasonable_fit
----------------------------------------
🏃 View run DecisionTree_training at: https://dagshub.com/mkhak23/ML_assignment2.mlflow/#/experiments/1/runs/7e1fe35d98c9470ba9011826a6959c94
🧪 View experiment at: https://dagshub.com/mkhak23/ML_assignment2.mlflow/#/experiments/1
{'max_depth': 8, 'min_samples_split': 200, 'min_samples_leaf': 50}
Train AUC: 0.8678067235971577
Val AUC: 0.859

,max_depth,min_samples_split,min_samples_leaf,val_roc_auc
2,8,200,50,0.859833
1,6,100,30,0.839969
0,4,50,20,0.802384


In [16]:
import mlflow
import mlflow.sklearn
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

best_params = results_df.iloc[0][[
    "max_depth",
    "min_samples_split",
    "min_samples_leaf"
]].to_dict()

best_params["max_depth"] = int(best_params["max_depth"])
best_params["min_samples_split"] = int(best_params["min_samples_split"])
best_params["min_samples_leaf"] = int(best_params["min_samples_leaf"])

X_full = train.drop(columns=["isFraud", "TransactionID"])
y_full = train["isFraud"]

final_model = DecisionTreeClassifier(
    **best_params,
    class_weight="balanced",
    random_state=42
)

final_pipeline = Pipeline([
    ("drop_useless", UselessFeatureDropper()),
    ("features", FraudFeatureEngineer()),
    ("cat_enc", CategoricalOrdinalEncoder()),
    ("imputer", DecisionTreeImputer()),
    ("model", final_model)
])

final_pipeline.fit(X_full, y_full)

mlflow.set_experiment("decision_tree_training")

with mlflow.start_run(run_name="final_best_decision_tree"):
    mlflow.log_params(best_params)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_metric("best_val_roc_auc", results_df.iloc[0]["val_roc_auc"])


    mlflow.sklearn.log_model(
        final_pipeline,
        name="final_pipeline"
    )

2026/05/05 09:58:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run final_best_decision_tree at: https://dagshub.com/mkhak23/ML_assignment2.mlflow/#/experiments/1/runs/fa620949672042859707123cf87148cc
🧪 View experiment at: https://dagshub.com/mkhak23/ML_assignment2.mlflow/#/experiments/1
